In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic_Train.csv')

print('Data Load')

Data Load


In [11]:
# First insepction of the data set
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [13]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [14]:
# Checking for missing values
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

**Note:** There are 177 missing values in the age column, 2 missing values in the Embarked column and almost all Cabin values are missing. To fill the missing age values, at first I will extract the title and will replace some specific title into rare. Afterwards I can group the dataset by title and fill the missing values with the median of each group, which is more accurate then filling all the missing values with the mean.

In [15]:
# Extract the title from the name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Checking if it works
print(df['Title'].unique())

['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'Countess' 'Jonkheer']


In [16]:
# Replacing 'Mme' 'Ms' 'Lady' and 'Mlle' into Mrs/Miss
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Lady', 'Mrs')

# Check the values again
print(df['Title'].value_counts())

Title
Mr          517
Miss        185
Mrs         127
Master       40
Dr            7
Rev           6
Major         2
Col           2
Don           1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


In [19]:
# Replacing the rare title into unknown by first creating a list with all the rare titles
unknown_title = ['Dr', 'Rev', 'Major', 'Col', 'Don', 'Sir', 'Capt', 'Countess', 'Jonkheer']

# Replacing into unknown
df['Title'] = df['Title'].replace(unknown_title, 'unknown')

# Check if it works
print(df['Title'].unique())

['Mr' 'Mrs' 'Miss' 'Master' 'unknown']


**Note:** Now I have 5 titles which I can group and calculate the median of each group. This way is much better for analysis, because the mean of all is 28 and this will distorts the data.

In [23]:
# Group the dataset by title and fill each group with the median of the group
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))

# Checking the the mean of each group
print(df.groupby('Title')['Age'].mean())

Title
Master      4.466750
Miss       21.681081
Mr         31.823017
Mrs        35.779528
unknown    45.590909
Name: Age, dtype: float64


In [24]:
# Checking the missing values again
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
Title            0
dtype: int64

In [ ]:
# Checking for the most frequent port with 'mode'
# mode is the most frequent value in that column
most_freq_port = df['Embarked'].mode()[0]

# Checking for the output of the most frequent port
print(most_freq_port)

S


In [30]:
# Filling the two missing 'Embarked' values with the most_freq_port
df['Embarked'] = df['Embarked'].fillna(most_freq_port)

# Check for missing values
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Title            0
dtype: int64